In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import pandas as pd
from keras import backend as K
from keras.models import Model
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.metrics import AUC, Precision, Recall
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
import pandas as pd
import pickle
import keras
from keras.saving import register_keras_serializable
from keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, concatenate, MultiHeadAttention, LayerNormalization, Add
import keras.backend as K
import keras_tuner as kt

In [ ]:
import tensorflow as tf
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Built with GPU support:", tf.test.is_built_with_gpu_support())
print("Available GPU devices:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Load train and test datasets
X_train_downsampled = pd.read_csv('data/readmit/X_train_downsampled.csv')
y_train_downsampled = pd.read_csv('data/readmit/y_train_downsampled.csv')
y_test = pd.read_csv('data/readmit/y_test.csv')
X_test = pd.read_csv('data/readmit/X_test.csv')

In [ ]:
# Load the LabelEncoder for ICD codes
with open('./Model/readmit_2016_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

In [ ]:
# Get the actual number of unique ICD codes
num_unique_icd_codes = len(encoder.classes_)
print("Number of unique ICD codes:", num_unique_icd_codes)

In [ ]:

@tf.keras.utils.register_keras_serializable(package="Custom")
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        config = super(TransformerBlock, self).get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class DeepSet(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, output_dim, **kwargs):
        super(DeepSet, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        # Element-wise transformation: phi network
        self.phi = tf.keras.Sequential([
            Dense(self.hidden_dim, activation='relu')
        ])
        # Post-aggregation transformation: rho network
        self.rho = tf.keras.Sequential([
            Dense(self.output_dim, activation='relu')
        ])

    def call(self, x):
        transformed = self.phi(x)  # (batch_size, num_codes, hidden_dim)
        aggregated = tf.reduce_sum(transformed, axis=1)  # (batch_size, hidden_dim)
        output = self.rho(aggregated)  # (batch_size, output_dim)
        return output

    def get_config(self):
        config = super(DeepSet, self).get_config()
        config.update({
            "input_dim": self.input_dim,
            "hidden_dim": self.hidden_dim,
            "output_dim": self.output_dim
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom")
def f2_score(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    epsilon = tf.keras.backend.epsilon()
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f2 = (5 * precision * recall) / (4 * precision + recall + epsilon)
    return f2

In [ ]:
class ICDHyperModel(kt.HyperModel):
    def build(self, hp):
        # Hyperparameters for ICD embedding and transformer/deep set
        embed_dim = hp.Int("embedding_size", 32, 128, step=32, default=32)  # fixed embedding size
        # Number of transformer blocks to stack (if you wish to use transformer blocks)
        num_transformer_blocks = hp.Int("num_transformer_blocks", 0, 3, default=0)
        transformer_ff_dim = hp.Int("transformer_ff_dim", 128, 256, step=32, default=128)
        transformer_dropout = hp.Float("transformer_dropout", 0.1, 0.5, step=0.1, default=0.4)
        
        # Hyperparameters for DeepSet aggregation (used here regardless of transformer use)
        deep_set_hidden_dim = hp.Int("deep_set_hidden_dim", 64, 256, step=32, default=128)
        deep_set_output_dim = hp.Int("deep_set_output_dim", 32, 128, step=32, default=64)
        
        # Hyperparameters for post-aggregation dense layers
        dense_units_1 = hp.Int("dense_units_1", 64, 256, step=32, default=128)
        dense_units_2 = hp.Int("dense_units_2", 32, 128, step=32, default=64)
        dense_units_3 = hp.Int("dense_units_3", 16, 64, step=16, default=32)
        dropout_rate = hp.Float("dense_dropout_rate", 0.2, 0.5, step=0.1, default=0.3)
        learning_rate = hp.Float("learning_rate", 1e-5, 1e-3, sampling="log", default=1e-4)
        
        # Define ICD-related inputs
        icd_columns = [f'I10_DX{i}' for i in range(1, 36)]
        icd_inputs = Input(shape=(len(icd_columns),), name='icd_codes')
        num_codes = num_unique_icd_codes  # Replace with your actual number
        icd_embedding = Embedding(
            input_dim=num_codes,
            output_dim=embed_dim,
            name='icd_embedding',
            trainable=True
        )(icd_inputs)
        
        x = icd_embedding

        # Optionally apply transformer blocks
        for i in range(num_transformer_blocks):
            transformer_block = TransformerBlock(embed_dim=embed_dim,
                                                 num_heads=5,
                                                 ff_dim=transformer_ff_dim,
                                                 rate=transformer_dropout)
            x = transformer_block(x, training=True)
        
        # Aggregate ICD embeddings via DeepSet (or you could choose sum/mean pooling)
        agg_block = DeepSet(input_dim=embed_dim, hidden_dim=deep_set_hidden_dim, output_dim=deep_set_output_dim)
        x = agg_block(x)
        
        # Define additional demographic and one-hot encoded inputs
        age_input = Input(shape=(1,), name='age')
        female_input = Input(shape=(1,), name='female')
        # Assuming these DataFrame filters return lists of column names:
        pay1_cols = [col for col in X_train_downsampled.columns if col.startswith('PAY1_')]
        zipinc_cols = [col for col in X_train_downsampled.columns if col.startswith('ZIPINC_QRTL_')]
        pay1_inputs = [Input(shape=(1,), name=col) for col in pay1_cols]
        zipinc_qrtl_inputs = [Input(shape=(1,), name=col) for col in zipinc_cols]
        
        # Concatenate all inputs
        concatenated = concatenate([x, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, name='concatenate')
        
        # Dense layers
        hidden = BatchNormalization(name='batch_norm')(concatenated)
        hidden = Dense(dense_units_1, activation='relu', kernel_regularizer=l2(0.001), name='dense_1')(hidden)
        hidden = Dropout(dropout_rate, name='dropout_1')(hidden)
        hidden = Dense(dense_units_2, activation='relu', kernel_regularizer=l2(0.001), name='dense_2')(hidden)
        hidden = Dropout(dropout_rate, name='dropout_2')(hidden)
        hidden = Dense(dense_units_3, activation='relu', kernel_regularizer=l2(0.001), name='dense_3')(hidden)
        hidden = Dropout(dropout_rate, name='dropout_3')(hidden)
        
        # Output layer for mortality prediction
        output = Dense(1, activation='sigmoid', name='output')(hidden)
        
        # Build and compile the model
        model = Model(inputs=[icd_inputs, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, outputs=output)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.binary_crossentropy,
            metrics=[AUC(name='auc'), Precision(name='precision'), Recall(name='recall'), f2_score]
        )
        return model

    def fit(self, hp, model, *args, **kwargs):
        # Tune batch size as well
        batch_size = hp.Choice("batch_size", [32, 64, 128], default=64)
        kwargs["batch_size"] = batch_size
        return model.fit(*args, **kwargs)

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPUs found:", gpus)
else:
    print("No GPU found. Training will use CPU.")


In [ ]:
import tensorflow as tf
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Built with GPU support:", tf.test.is_built_with_gpu_support())


In [ ]:
early_stopping = EarlyStopping(
    monitor='val_f2_score', patience=3, mode='max', restore_best_weights=True
)
tf.debugging.set_log_device_placement(True)

# Create an instance of the HyperModel
hypermodel = ICDHyperModel()

# Initialize the tuner (here using RandomSearch; you can also try Hyperband or BayesianOptimization)
tuner = kt.RandomSearch(
    hypermodel,
    objective=kt.Objective("val_f2_score", direction="max"),
    max_trials=10,
    executions_per_trial=1,
    directory="my_dir",
    project_name="ICD_hyperparameter_tuning"
)

# Prepare your training data.
# Note: X_train_downsampled is assumed to be your DataFrame containing the inputs.
# Ensure that the columns match the input names defined in the hypermodel.
icd_columns = [f'I10_DX{i}' for i in range(1, 36)]
pay1_cols = [col for col in X_train_downsampled.columns if col.startswith('PAY1_')]
zipinc_cols = [col for col in X_train_downsampled.columns if col.startswith('ZIPINC_QRTL_')]

train_inputs = [X_train_downsampled[icd_columns],
                X_train_downsampled['AGE'],
                X_train_downsampled['FEMALE']] + \
               [X_train_downsampled[col] for col in pay1_cols] + \
               [X_train_downsampled[col] for col in zipinc_cols]

# Run the hyperparameter search
tuner.search(
    train_inputs,
    y_train_downsampled,
    epochs=10,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1
)

# Retrieve the best model.
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
# Step 4: Evaluate the model
test_loss, test_auc, test_precision, test_recall, test_f2 = best_model.evaluate(
    [X_test[icd_columns], X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in X_test.filter(regex='PAY1_').columns] + [X_test[col] for col in X_test.filter(regex='ZIPINC_QRTL_').columns], 
    y_test)

print(f'Test AUC: {test_auc:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall: {test_recall:.4f}')
print(f'Test F2 Score: {test_f2:.4f}')


In [ ]:
# # Save the trained model
model.save('Model/readmit_2016_MLP_deepset.keras')

# keras.saving.save_model(model, 'Model/mort_2016.keras')